In [1]:
import torchvision
from torchvision import transforms
import torchvision.datasets as datasets
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import torchvision.utils as vutils

from denoising_diffusion_pytorch import Unet, GaussianDiffusion

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

LEARNING_RATE = 4e-4
BATCH_SIZE = 512  
N_EPOCHS = 100
IMAGE_SIZE = 28
TIME_STEPS = 1000
SAMPLING_TIMESTEPS = 250

myTransforms = transforms.Compose([transforms.ToTensor()]) ############


print("loading MNIST digits dataset")
dataset = datasets.MNIST(root="dataset/", transform=myTransforms, download=True)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = datasets.MNIST(root='dataset/', train=False, download=False, transform=myTransforms)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

DIM = 32
DIM_MULTS = (1, 2, 5)
model = Unet(dim = DIM, dim_mults = DIM_MULTS, flash_attn = False, channels = 1)




diffusion = GaussianDiffusion(
    model,
    image_size = IMAGE_SIZE,
    timesteps = TIME_STEPS,         
    sampling_timesteps = SAMPLING_TIMESTEPS   
)


model.to(device)
diffusion.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

weights_path = "ddpm_mnist_model.pth"
try:
   
    model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
    print("Successfully loaded saved weights.")
except FileNotFoundError:
    print(f"No saved weights found at {weights_path}. Starting from scratch.")

c:\Users\J Birbou\ML_Class\ml_env6\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
loading MNIST digits dataset
Successfully loaded saved weights.


In [ ]:


# Training loop
epochs = tqdm(range(N_EPOCHS)) 
for e in epochs: 
    model.train() 
    total_train_loss = 0.0
    
    
    for step, (images, labels) in enumerate(loader): #######
        
        images = images.to(device)
        
        optimizer.zero_grad()
        
        
        loss = diffusion(images) 
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        

    
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad(): 
        for val_images, _ in test_loader:
            val_images = val_images.to(device)
            val_loss = diffusion(val_images)
            total_val_loss += val_loss.item()
            
    avg_train_loss = total_train_loss / len(loader)
    avg_val_loss = total_val_loss / len(test_loader)

    epochs.set_postfix(
        train_loss=f"{avg_train_loss:.4f}",
        val_loss=f"{avg_val_loss:.4f}"
    )

 34%|███▍      | 34/100 [18:20<35:34, 32.34s/it, train_loss=0.0333, val_loss=0.0349]

In [ ]:
#Save weights
torch.save(model.state_dict(), "ddpm_mnist_model.pth")
print("Model weights saved to ddpm_mnist_model.pth")


Model weights saved to ddpm_mnist_model.pth
Generating new handwritten digits...


sampling loop time step: 100%|██████████| 250/250 [00:04<00:00, 61.16it/s]

Generated images saved to generated_mnist_digits.png


In [ ]:
print("Generating new handwritten digits...")
model.eval()      
with torch.no_grad():
    final_sampled_images = diffusion.sample(batch_size=16)

vutils.save_image(final_sampled_images, "generated_mnist_digits.png", nrow=4)
print("Generated images saved to generated_mnist_digits.png")           